# What does AntiFold actually write, and what is in it?

This notebook exists to put one tool's real output on screen before anything in
this package is taught to read it. AntiFold is a default scorer here and the
only one whose command line has been checked against its own README, but
nothing in the repo has ever seen a byte it produced. Until that is true,
`antifold_process` is a plausible guess with a `verified=True` next to it.

**The rule this notebook follows: no parser before the output is on screen.**
Part 5 is deliberately empty on first run. Parts 1 to 4 are a measurement — the
structure, the invocation, the raw bytes, and a verbatim copy of them kept as a
fixture. Only once the fixture exists does writing a reader stop being
guesswork, because from then on the reader can be tested offline forever
against bytes the tool really wrote.

| Part | What it does | Reads | Writes |
|---|---|---|---|
| 1 | Survey the local 1N8Z: chains, lengths, where VH ends | the PDB | nothing |
| 2 | Run AntiFold once, with the checked flags | — | `OUT_DIR` |
| 3 | Print every output file raw, 30 lines, unparsed | `OUT_DIR` | nothing |
| 4 | Copy it verbatim to `tests/fixtures/antifold_1n8z/` and commit | `OUT_DIR` | the fixture |
| 5 | Convert to `LogProbMatrix` — **empty until Part 3 has run** | the fixture | nothing |

Run it on the GPU box, in the environment that holds this package (Env A), with
`ANTIFOLD_PYTHON` pointing at the environment that holds AntiFold (Env B). The
two cannot be the same environment: `pip install antifold` pins
`biotite==0.38` and this package needs `biotite>=2.0`, which pip calls
`ResolutionImpossible`.

It imports `design_pipeline` to read the structure — the chain sequence and the
file's own residue labels, which are the two numberings the alignment check
turns on — and modifies nothing in it. Once Part 5 works, the function moves
into the package with a fixture test. Not before.

In [ ]:
# --- parameters -------------------------------------------------------------

# The local 1N8Z. None searches the places a copy usually lands (printed below
# if the search fails). Nothing here downloads: a structure that arrived by a
# different route is a different structure, and this notebook is about one file.
PDB_PATH = None

# 1N8Z is trastuzumab Fab bound to HER2 domain IV. The heavy chain is 'B', not
# 'H' - exactly the labelling trap the harness checks `chains` for - so these
# are asserted against the file in Part 1 rather than trusted.
HEAVY_CHAIN = "B"
LIGHT_CHAIN = "A"
ANTIGEN_CHAIN = "C"

# Env B's interpreter: the one that can `import antifold`. DESIGN_PIPELINE_ANTIFOLD
# is the variable the package's own ProcessAdapter reads, so setting it once in
# the shell configures both this notebook and `score_variants`.
ANTIFOLD_PYTHON = None          # None -> $DESIGN_PIPELINE_ANTIFOLD, then "python"

OUT_DIR = "runs/antifold_capture"      # AntiFold's --out_dir
FIXTURE_DIR = "tests/fixtures/antifold_1n8z"
HEAD_LINES = 30                        # raw lines printed per output file

# A bare VH is about this long. The point of comparison for a Fab heavy chain,
# which carries CH1 on top of it; our scFv is VH + linker + VL, around 240 aa
# in total (docs/research/04_adaptyv_rules_and_assay.md).
SCFV_VH_LENGTH = 120

COMMIT_FIXTURE = True           # Part 4 runs `git add` and `git commit`

In [ ]:
import hashlib
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

from design_pipeline import numbering, structures

CANDIDATES = (
    Path("data/structures/1N8Z.pdb"),
    Path("data/1N8Z.pdb"),
    Path.home() / ".cache" / "design_pipeline" / "pdb" / "1n8z.pdb",
)


def resolve_pdb() -> Path:
    if PDB_PATH is not None:
        path = Path(PDB_PATH).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f"PDB_PATH does not exist: {path}")
        return path
    for candidate in CANDIDATES:
        if candidate.is_file():
            return candidate
    # Named, every one of them: a notebook that says "not found" and makes you
    # guess where it looked costs more than these four lines.
    looked = "\n".join(f"  {c}" for c in CANDIDATES)
    raise FileNotFoundError(
        f"no local 1N8Z found. Looked in:\n{looked}\n"
        f"Set PDB_PATH to the copy you mean to use."
    )


pdb = resolve_pdb()
out_dir = Path(OUT_DIR)
antifold_python = (
    ANTIFOLD_PYTHON or os.environ.get("DESIGN_PIPELINE_ANTIFOLD") or "python"
)

print(f"structure:       {pdb}  ({pdb.stat().st_size:,} bytes)")
print(f"out_dir:         {out_dir.resolve()}")
print(f"antifold python: {antifold_python}")
print(f"this python:     {sys.executable}")
print(f"anarci:          {'available' if numbering.available() else 'NOT INSTALLED'}")

## Part 1 — the structure

Three things, in this order: what chains the file holds, which of them is
heavy, light and antigen, and how much of the heavy chain is *not* variable
domain. The last one is the question that decides whether anything downstream
can use this file as-is.

Two numberings appear below and they are not the same. `chain_sequence` is the
chain in file order, 1-based, and that is what `position` means everywhere in
this package's variant tables. `chain_residue_ids` is the file's own labels —
which skip, repeat and carry insertion codes like `100A` — and that is what
AntiFold reports, because it reports the PDB's own numbering. Every mapping
between the two has to be explicit or it is wrong in a way that still produces
plausible numbers.

In [ ]:
chains = structures.chain_ids(pdb)
print(f"chains with amino acids, in file order: {', '.join(chains)}\n")

sequences: dict[str, str] = {}
residue_ids: dict[str, tuple[str, ...]] = {}

print(f"{'chain':<6}{'len':>5}  {'first':<14}{'last':<14}")
print("-" * 40)
for chain in chains:
    sequence = structures.chain_sequence(pdb, chain)
    labels = structures.chain_residue_ids(pdb, chain)
    sequences[chain] = sequence
    residue_ids[chain] = labels
    first = f"{sequence[0]} ({labels[0]})"
    last = f"{sequence[-1]} ({labels[-1]})"
    print(f"{chain:<6}{len(sequence):>5}  {first:<14}{last:<14}")

print()
for chain in chains:
    sequence = sequences[chain]
    print(f"chain {chain} ({len(sequence)} aa)")
    print(f"  {sequence[:60]}")
    if len(sequence) > 60:
        print(f"  ... {sequence[-30:]}")

# Does the file's own numbering coincide with our 1-based chain positions? For
# some chains it does and for some it does not, and the difference decides
# whether AntiFold's reported numbering can be used as a position directly or
# has to be looked up through `chain_residue_ids`. Measured, not assumed.
print()
print(f"{'chain':<7}{'labels':<14}{'insertions':>11}{'gaps':>6}{'1-based and contiguous?':>26}")
print("-" * 64)
for chain in chains:
    labels = residue_ids[chain]
    insertions = sum(1 for label in labels if not label.isdigit())
    numeric = [int(label) for label in labels if label.isdigit()]
    gaps = 0
    for earlier, later in zip(numeric, numeric[1:]):
        if later - earlier > 1:
            gaps += later - earlier - 1
    contiguous = (
        not insertions and numeric == list(range(1, len(labels) + 1))
    )
    print(
        f"{chain:<7}{labels[0] + '-' + labels[-1]:<14}{insertions:>11}{gaps:>6}"
        f"{('yes' if contiguous else 'NO'):>26}"
    )
print()
print("A 'yes' row means position == PDB label for that chain, by coincidence of")
print("how this file was deposited - not by rule, and not something to rely on")
print("for the next structure. A 'NO' row means the two numberings differ and")
print("every mapping between them has to go through chain_residue_ids.")

In [ ]:
# The chain roles, asserted rather than assumed. A labelling mistake here would
# be handed straight to AntiFold's --heavy_chain/--light_chain flags, and a
# wrong-but-valid chain letter is the kind of error that returns 0 and a CSV.
missing = [
    (role, chain)
    for role, chain in (
        ("heavy", HEAVY_CHAIN), ("light", LIGHT_CHAIN), ("antigen", ANTIGEN_CHAIN)
    )
    if chain not in chains
]
if missing:
    raise KeyError(
        f"chains named in the parameters that this file does not have: "
        f"{', '.join(f'{role}={chain!r}' for role, chain in missing)}. "
        f"Present: {', '.join(chains)}"
    )

# IMGT numbering is the independent check on the roles: ANARCI labels a variable
# domain and nothing else, so the antigen should come back with no labels at all
# and each antibody chain with ~120.
labels_imgt = {chain: numbering.imgt_labels(sequences[chain]) for chain in chains}

print(f"{'chain':<7}{'role':<9}{'len':>5}{'IMGT-labelled':>15}{'VH/VL span':>14}")
print("-" * 50)
spans: dict[str, tuple[int, int] | None] = {}
for chain in chains:
    role = {HEAVY_CHAIN: "heavy", LIGHT_CHAIN: "light", ANTIGEN_CHAIN: "antigen"}.get(
        chain, "-"
    )
    labelled = [i for i, label in enumerate(labels_imgt[chain]) if label]
    if labelled:
        span = (labelled[0] + 1, labelled[-1] + 1)   # 1-based chain positions
        spans[chain] = span
        span_text = f"{span[0]}-{span[1]}"
    else:
        spans[chain] = None
        span_text = "none"
    print(
        f"{chain:<7}{role:<9}{len(sequences[chain]):>5}"
        f"{len(labelled):>15}{span_text:>14}"
    )

if not numbering.available():
    print()
    print("ANARCI is not installed, so the spans above are empty for every chain")
    print("and the variable-domain boundary below cannot be measured. Install it")
    print("(`uv pip install 'design-pipeline[numbering]'`, needs HMMER) before")
    print("reading the next cell as a result.")

In [ ]:
# How much of the heavy chain is CH1 - i.e. how far this Fab is from the scFv
# the campaign actually designs.
heavy = sequences[HEAVY_CHAIN]
span = spans[HEAVY_CHAIN]

print(f"heavy chain {HEAVY_CHAIN}: {len(heavy)} residues "
      f"({residue_ids[HEAVY_CHAIN][0]} to {residue_ids[HEAVY_CHAIN][-1]} "
      f"in the file's own numbering)")

# Where the variable domain ends. ANARCI is the general answer and the one to
# trust across antibodies. Without it, one measurement is still available on
# this particular file: the N-terminal motif of a human IgG1 CH1 domain.
# Finding it is a landmark lookup in one sequence, not a numbering scheme, and
# it is reported as exactly that - the position the string occurs at.
CH1_MOTIF = "ASTKGPSVFPLAP"

if span is not None:
    vh_start, vh_end = span
    boundary_from = "ANARCI (IMGT numbering)"
else:
    found = heavy.find(CH1_MOTIF)
    vh_start, vh_end = (1, found) if found > 0 else (None, None)
    boundary_from = (
        f"CH1 landmark {CH1_MOTIF!r} at position {found + 1} - no ANARCI"
        if found > 0
        else "nothing: no ANARCI, and the CH1 landmark is not in this chain"
    )

print(f"boundary from:   {boundary_from}")

if vh_end is None:
    print("variable domain: NOT MEASURED. Install anarci (`uv pip install")
    print("'design-pipeline[numbering]'`, needs HMMER) and rerun before reading")
    print("anything below as a result.")
else:
    print(f"variable domain: positions {vh_start}-{vh_end} "
          f"(file labels {residue_ids[HEAVY_CHAIN][vh_start - 1]} to "
          f"{residue_ids[HEAVY_CHAIN][vh_end - 1]}), {vh_end - vh_start + 1} residues")
    print(f"constant region: positions {vh_end + 1}-{len(heavy)}, "
          f"{len(heavy) - vh_end} residues - this is CH1")
    print()
    print(f"so this chain is {len(heavy) / vh_end:.2f}x the length of its own variable")
    print(f"domain: {len(heavy)} residues against {vh_end}. A bare VH is about")
    print(f"{SCFV_VH_LENGTH} aa, which the measured {vh_end} agrees with - so what the Fab")
    print(f"adds over the scFv's VH is exactly the {len(heavy) - vh_end} residues of CH1,")
    print(f"{100 * (len(heavy) - vh_end) // len(heavy)}% of the chain, and every one of")
    print("them is a position a full scan would mutate.")

In [ ]:
# Can enumerate_variants map onto this file as-is? Run it rather than argue
# about it. Restricted to one replacement residue, so the parquet stays small;
# the question is whether the positions line up, not how many rows there are.
from design_pipeline.structures import Complex
from design_pipeline.variants import enumerate_variants

complex_ = Complex(paths=(pdb,), output_dir=out_dir, config={"source": str(pdb)})

full_scan = sum(19 for residue in heavy if residue != "X")
print(f"a full scan of chain {HEAVY_CHAIN} is {full_scan:,} candidates")
if vh_end is not None:
    variable_only = sum(19 for residue in heavy[:vh_end] if residue != "X")
    print(f"the variable domain alone is {variable_only:,} "
          f"({100 * variable_only // full_scan}% of them)")

try:
    table = enumerate_variants(
        complex_, chain=HEAVY_CHAIN, mutations="A",
        positions=range(1, len(heavy) + 1),
    )
except Exception as error:
    print(f"\nenumerate_variants raised {type(error).__name__}: {error}")
    table = None
else:
    frame = table.frame
    labelled_rows = int((frame["imgt"].astype(str) != "").sum())
    print(f"\nenumerate_variants returned {len(frame):,} rows over "
          f"{frame['position'].nunique()} positions")
    print(f"  rows carrying an IMGT label: {labelled_rows:,} "
          f"({len(frame) - labelled_rows:,} without one)")
    print(f"  position range: {frame['position'].min()}-{frame['position'].max()}")
    display(frame.head(3))
    display(frame.tail(3))

In [ ]:
# The verdict, stated plainly, from the numbers above and nothing else.
print("Does enumerate_variants map onto this structure as-is?")
print()
print("Mechanically, yes. `position` is a 1-based index into the chain in file")
print("order, so it is defined for every residue of a Fab heavy chain exactly as")
print("it is for an scFv's VH; nothing about CH1 makes it raise, and the rows")
print("above prove it.")
print()
if vh_end is not None:
    print(f"Usefully, no - not with the default scan. {len(heavy) - vh_end} of the")
    print(f"{len(heavy)} positions are CH1, so a default `enumerate_variants(complex_,")
    print(f"chain='{HEAVY_CHAIN}')` spends {100 * (full_scan - variable_only) // full_scan}% "
          f"of its {full_scan:,} candidates mutating a")
    print("constant domain that is not part of the molecule being designed. Those")
    print("rows carry an empty `imgt`, which is the visible symptom: no IMGT label")
    print("means the position does not transfer to another antibody, which is the")
    print("whole reason for scoring it.")
    print()
    print(f"  positions=range(1, {vh_end + 1})   # the variable domain, this file only")
    print()
    print("The position numbers are file-specific, so that range is not portable.")
    print("Deriving it from ANARCI per structure is, which is what the span above is.")
print("Still open, and Part 3 decides it: whether AntiFold emits rows for CH1 and")
print("for the antigen at all. If it scores only the variable domains, the matrix")
print("will not cover every row this table can produce, and the uncovered ones")
print("have to come back as None rather than as zero.")

## Part 2 — run AntiFold once

The four flags the package's `antifold_process` adapter is built from, plus the
chain flags: `--pdb_file`, `--out_dir`, `--heavy_chain`, `--light_chain`,
`--antigen_chain`. Nothing else, no batch size, no sampling options. One
invocation of the documented command line on one file.

The command, the exit code and the whole of stdout and stderr are printed. No
`check=True` and nothing swallowed: if this fails, the failure is the result,
and the cell prints it and then stops the notebook rather than carrying on to
produce something from an empty directory. Weights download on first run, so
the first call is slow and the download noise belongs in the log below.

In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)

command = [
    antifold_python, "-m", "antifold.main",
    "--pdb_file", str(pdb),
    "--out_dir", str(out_dir),
    "--heavy_chain", HEAVY_CHAIN,
    "--light_chain", LIGHT_CHAIN,
    "--antigen_chain", ANTIGEN_CHAIN,
]

print("$ " + " ".join(shlex.quote(part) for part in command))
print()

completed = subprocess.run(command, capture_output=True, text=True)

print(f"exit code: {completed.returncode}")
print()
print("--- stdout " + "-" * 60)
print(completed.stdout if completed.stdout else "(empty)")
print("--- stderr " + "-" * 60)
print(completed.stderr if completed.stderr else "(empty)")
print("-" * 71)

In [ ]:
written = sorted(p for p in out_dir.rglob("*") if p.is_file())

print(f"{len(written)} file(s) under {out_dir}:")
for path in written:
    print(f"  {path.relative_to(out_dir).as_posix():<50}{path.stat().st_size:>12,} bytes")
if not written:
    print("  (nothing)")

if completed.returncode != 0 or not written:
    # The instruction this notebook was written under: a failure is as useful a
    # result as a success, and working around it would throw that result away.
    raise RuntimeError(
        f"AntiFold exited {completed.returncode} and wrote {len(written)} file(s). "
        f"The command, the exit code and both streams are printed above; that is "
        f"the finding. Stop here - do not work around it."
    )

## Part 3 — look at the output

First the raw bytes, decoded and printed, before anything in this notebook has
an opinion about what they mean. Then measurements that count characters rather
than interpret them — line counts, delimiter counts, the first line as it is.
Then, in a markdown cell, the description: written by hand after reading the
output above, not generated.

Nothing here splits a line into fields. The moment it does, the format has been
assumed, and the point of this part is to find out whether the assumption would
have been right.

In [ ]:
for path in written:
    relative = path.relative_to(out_dir).as_posix()
    print("=" * 71)
    print(f"{relative}   ({path.stat().st_size:,} bytes)")
    print("=" * 71)
    try:
        text = path.read_text(encoding="utf-8")
    except UnicodeDecodeError as error:
        print(f"not UTF-8 text: {error}")
        print(f"first 200 bytes, repr: {path.read_bytes()[:200]!r}")
        continue
    lines = text.splitlines()
    for number, line in enumerate(lines[:HEAD_LINES], start=1):
        print(f"{number:>4} | {line}")
    if len(lines) > HEAD_LINES:
        print(f"     | ... {len(lines) - HEAD_LINES:,} more line(s)")
    print()

In [ ]:
# Measurements, not interpretation. Counting a comma is not parsing a CSV: none
# of this decides what a column means, it only says what the file is shaped
# like. Line 1 is reported against the most common count over the head: when
# the two disagree, line 1 is not a data row - a header, or a comment preamble
# - and that is worth seeing before anything tries to read the file.
from collections import Counter

TAB = chr(9)


def modal(counts: list[int]) -> int:
    return Counter(counts).most_common(1)[0][0] if counts else 0


print("counts read 'line 1 / most common over the first 50 lines'")
print()
print(f"{'file':<34}{'lines':>8}{'commas':>12}{'tabs':>10}")
print("-" * 64)
for path in written:
    try:
        text = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        print(f"{path.relative_to(out_dir).as_posix():<34}{'not text':>8}")
        continue
    lines = text.splitlines()
    head = lines[:50]
    first = lines[0] if lines else ""
    commas = f"{first.count(',')}/{modal([line.count(',') for line in head])}"
    tabs = f"{first.count(TAB)}/{modal([line.count(TAB) for line in head])}"
    print(
        f"{path.relative_to(out_dir).as_posix():<34}{len(lines):>8}"
        f"{commas:>12}{tabs:>10}"
    )

print()
print("Chain letters anywhere in each file (a cheap check on coverage - it says")
print("the letter occurs, not that the chain has rows):")
for path in written:
    try:
        text = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        continue
    present = [
        f"{role}={chain}"
        for role, chain in (
            ("H", HEAVY_CHAIN), ("L", LIGHT_CHAIN), ("Ag", ANTIGEN_CHAIN)
        )
        if f",{chain}," in text or f"{TAB}{chain}{TAB}" in text
    ]
    print(f"  {path.relative_to(out_dir).as_posix():<40}{', '.join(present) or 'none'}")

### What the output actually looks like

**Fill this in from the run above. Nothing here is guessed in advance** - that
is the whole discipline of this notebook, and a description written before the
run would be the parser-before-the-output mistake wearing prose.

Answer these, and say "ambiguous" where it is ambiguous rather than picking the
likely reading:

1. **Format.** One file or several? Delimiter? Is there a header line, a
   comment preamble, both? (The measurement cell above flags a line 1 whose
   field count differs from the rest.)
2. **Columns.** Their names, in order. Which carries the position, which the
   chain, which the wild-type residue.
3. **Residue numbering.** Is it the PDB's own labels or 1-based into the chain?
   Are insertion codes present (`100A`), and if so in their own column or
   appended to the number? Part 1 measured that chains A and B of this file are
   numbered 1..n contiguously while chain C is not, so for the antibody chains
   the two numberings coincide **in this file** and a parser that relies on
   that will break on the next structure. This is the field that decides
   whether the mapping onto `position` has to go through `chain_residue_ids`.
4. **The 20 amino acids.** Twenty columns, one per residue, in what order - the
   file's own order, not assumed to be `ACDEFGHIKLMNPQRSTVWY`? Or long form,
   one row per (position, residue)? Are the values log probabilities,
   probabilities, or log-likelihood ratios against the wild type, and is that
   stated anywhere in the file or only inferred from their range?
5. **Coverage.** Heavy, light, antigen - which chains have rows? Does the heavy
   chain's row count come to 220, its full length including CH1, or to 120, the
   variable domain alone? Is every position present, or are there gaps?
6. **Anything unexplained.** A column whose meaning is not obvious from the
   file is a question for AntiFold's own documentation, recorded here as open
   rather than resolved by assumption.

## Part 4 - capture the fixture

Copy the output verbatim into `tests/fixtures/antifold_1n8z/` and commit it.
This is the durable artifact of the whole exercise: once these bytes are in the
repository, a reader for them can be tested on a laptop with no GPU, no Env B
and no network, forever, against output the tool really produced. Without it,
every future change to the parser needs this box again.

`copy2`, byte for byte, with a SHA-256 printed per file so the copy can be
shown to be a copy. The files are small text; if AntiFold wrote anything large
or binary it is listed in the sizes above and should be left out by hand rather
than committed by habit.

In [ ]:
fixture_dir = Path(FIXTURE_DIR)
fixture_dir.mkdir(parents=True, exist_ok=True)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


copied: list[Path] = []
print(f"{'file':<40}{'bytes':>10}  sha256")
print("-" * 71)
for path in written:
    relative = path.relative_to(out_dir)
    target = fixture_dir / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, target)
    digest = sha256(target)
    assert digest == sha256(path), f"{relative}: copy does not match the original"
    copied.append(target)
    print(f"{relative.as_posix():<40}{target.stat().st_size:>10,}  {digest[:16]}...")

# What was captured, in a form a test can assert against without re-reading the
# directory listing: the provenance of the bytes, beside the bytes.
provenance = fixture_dir / "PROVENANCE.txt"
provenance.write_text(
    "AntiFold output captured verbatim by notebooks/02_antifold_capture.ipynb\n"
    f"structure:  {pdb}\n"
    f"chains:     heavy={HEAVY_CHAIN} light={LIGHT_CHAIN} antigen={ANTIGEN_CHAIN}\n"
    f"command:    {' '.join(shlex.quote(part) for part in command)}\n"
    f"exit code:  {completed.returncode}\n"
    f"files:\n"
    + "".join(
        f"  {p.relative_to(fixture_dir).as_posix()}  "
        f"{p.stat().st_size} bytes  sha256={sha256(p)}\n"
        for p in copied
    ),
    encoding="utf-8",
)
print(f"\nwrote {provenance}")

In [ ]:
message = (
    "Capture AntiFold's real 1N8Z output as a test fixture\n"
    "\n"
    "The bytes AntiFold actually writes, kept verbatim so a reader for them can "
    "be tested offline - no GPU, no Env B, no network. Produced by "
    "notebooks/02_antifold_capture.ipynb; the command and per-file hashes are in "
    "PROVENANCE.txt.\n"
    "\n"
    "Co-Authored-By: Claude Opus 5 <noreply@anthropic.com>\n"
)

add = ["git", "add", "--", str(fixture_dir)]
commit = ["git", "commit", "-m", message]

if not COMMIT_FIXTURE:
    print("COMMIT_FIXTURE is False. Run these yourself:")
    print("$ " + " ".join(shlex.quote(part) for part in add))
    print("$ git commit -F - <<'EOF'\n" + message + "EOF")
else:
    for argv in (add, commit):
        print("$ " + " ".join(shlex.quote(part) for part in argv))
        result = subprocess.run(argv, capture_output=True, text=True)
        print(result.stdout or "", end="")
        print(result.stderr or "", end="")
        if result.returncode != 0:
            print(f"\ngit exited {result.returncode}; commit the fixture by hand.")
            break
    else:
        print()
        print(subprocess.run(
            ["git", "log", "-1", "--stat", "--oneline"],
            capture_output=True, text=True,
        ).stdout)

## Part 5 — parse, in the notebook

**Empty by design until Part 3 has run.** Everything above is a measurement;
this is the first cell that would encode a belief about the format, and writing
it now — from AntiFold's README, from how ProteinMPNN writes its scores, from
what a per-residue log-probability file usually looks like — is exactly the
mistake the notebook is arranged to prevent. A parser built on a guess that
happens to be nearly right is worse than one that fails: it returns real
log-odds for the wrong positions, and `score = M[pos, mut] - M[pos, wt]` is a
plausible number whichever row it came off.

When the raw output above is on screen, the conversion goes in the next cell,
and these are the checks it has to pass before it is believed. They are written
now because they do not depend on the format:

1. **The wild type the matrix implies at every position matches the PDB.**
   `LogProbMatrix` carries the sequence its rows describe, and
   `assert_matches(chain_sequence(pdb, chain))` is this check already written —
   it reports the first differing position rather than a boolean. If AntiFold
   names a wild-type residue per row, that column must agree with the sequence
   too, independently: two sources of the same truth, which is what catches an
   off-by-one that happens to land on an identical residue.
2. **Row count matches chain length.** `len(matrix.sequence) == len(chain_sequence(...))`
   for the rows covered, with `offset` carrying how far in the first row sits.
   A matrix covering only the variable domain is legitimate and takes an
   offset; one covering fewer rows than it claims is not.
3. **Three positions checked by hand against the raw file.** One in a CDR, one
   in a framework position, one at a chain boundary — the first or last residue
   AntiFold reports. Read the number out of the text file with your eyes, print
   `matrix.log_prob(position, residue)`, and compare. Automated round-trips
   through the same parser cannot catch a systematic misreading; this is the
   only check here that can.
4. **Chain coverage is explicit.** One matrix per chain AntiFold scored, and
   nothing invented for a chain it did not. A position with no row reads back as
   `None` from `log_prob`, never as `0.0` — a zero is a log-probability of 1.0,
   the most confident claim the scale can make.

Then report the function here. It moves into `design_pipeline/` afterwards,
with a test that reads `tests/fixtures/antifold_1n8z/` — not before.

In [ ]:
raise NotImplementedError(
    "Part 5 is written after Part 3 has run, against the output printed there. "
    "Run Parts 1-4 on the GPU box, read the raw output, fill in the description "
    "cell, then write the conversion here."
)